# 06 - NaN Subcategory Deep Dive

All analyses of grievances whose `subcategory` is missing (`NaN` or empty
string).

Shared scaffolding is in `grievance_common.py`.

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gpd

from grievance_common import (
    load_complaints, add_time_fields, filter_window, population_df,
    load_district_shapefile, draw_choropleth,
    OUTPUT_DIR, FOCUS_YEAR, VALID_YEARS,
)

# IPython display() is auto-available in Jupyter; fall back outside it.
try:
    from IPython.display import display
except ImportError:
    display = print

## Data Load

Load the complaints table, derive time fields, apply the analysis window.
Two NaN definitions are useful: the dataset has both true-null subcategories
and blank strings; we treat both as NaN.

In [2]:
df = add_time_fields(load_complaints())
df = filter_window(df)

# NaN filter: subcategory is missing OR blank string.
nan_mask = df["subcategory"].isna() | (
    df["subcategory"].astype(str).str.strip().isin(["", "NaN"])
)
print(f"Total complaints in window : {len(df):,}")
print(f"With NaN subcategory       : {int(nan_mask.sum()):,}")
print(f"NaN share                  : {nan_mask.mean()*100:.2f}%")

Total complaints in window : 1,323,840
With NaN subcategory       : 197,952
NaN share                  : 14.95%


In [4]:
df["year_month"] = df["created_on"].dt.to_period("M").astype(str)
df["district"]   = df["district"].fillna("NaN").astype(str).str.strip()
df["block"]      = df["block"].fillna("NaN").astype(str).str.strip()
if "petitioner_name" in df.columns:
    df["petitioner_name"] = (df["petitioner_name"].fillna("NaN")
                             .astype(str).str.strip())
 

## Helper Functions

In [5]:
# ----- NaN identification ----------------------------------------------------
 
def nan_mask(frame, col="subcategory"):
    """True where col is null or blank/NaN string."""
    return frame[col].isna() | (
        frame[col].astype(str).str.strip().isin(["", "NaN"]))
 
 
# ----- count + NaN + pct -----------------------------------------------------
 
def nan_dist_table(frame, group_cols, nan_frame=None):
    """Total complaints, NaN complaints and NaN pct by group_cols."""
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if nan_frame is None:
        nan_frame = frame[nan_mask(frame)]
    total = (frame.groupby(group_cols)["ticket_no"]
             .nunique().reset_index(name="total_complaints"))
    nan_c = (nan_frame.groupby(group_cols)["ticket_no"]
             .nunique().reset_index(name="nan_complaints"))
    out = total.merge(nan_c, on=group_cols, how="left")
    out["nan_complaints"] = out["nan_complaints"].fillna(0).astype(int)
    out["nan_pct"] = (
        out["nan_complaints"] / out["total_complaints"] * 100).round(2)
    return out
 
 
def nan_dist_with_aggregate(frame, group_col, nan_frame=None):
    """nan_dist_table by group_col x year + Aggregate column.
    Returns (long table, wide pivot)."""
    if nan_frame is None:
        nan_frame = frame[nan_mask(frame)]
    long = nan_dist_table(frame, [group_col, "custom_year"],
                          nan_frame=nan_frame)
    agg  = nan_dist_table(frame, [group_col], nan_frame=nan_frame)
    agg["custom_year"] = "Aggregate"
    long_full  = pd.concat([long, agg], ignore_index=True)
    year_order = VALID_YEARS + ["Aggregate"]
    pivot = long_full.pivot_table(
        index=group_col, columns="custom_year",
        values=["total_complaints", "nan_complaints", "nan_pct"],
        aggfunc="sum")
    pivot = pivot.swaplevel(0, 1, axis=1)
    pivot = pivot.reindex(
        columns=pd.MultiIndex.from_product(
            [year_order,
             ["total_complaints", "nan_complaints", "nan_pct"]]),
        fill_value=0)
    for col in pivot.columns:
        if col[1] == "nan_pct":
            pivot[col] = pivot[col].round(2)
        else:
            pivot[col] = pivot[col].fillna(0).astype(int)
    return long_full, pivot
 
 
def nan_dist_pivot(long_table, index_col, col_col,
                   values=("total_complaints", "nan_complaints", "nan_pct")):
    """Wide pivot of a nan_dist_table (for focus-year month tables)."""
    pivot = long_table.pivot_table(
        index=index_col, columns=col_col,
        values=list(values), aggfunc="sum")
    pivot = pivot.swaplevel(0, 1, axis=1).sort_index(axis=1)
    for col in pivot.columns:
        if col[1] == "nan_pct":
            pivot[col] = pivot[col].round(2)
        else:
            pivot[col] = pivot[col].fillna(0).astype(int)
    return pivot
 
 
# ----- per-capita ------------------------------------------------------------
 
def add_per_capita(table, pop_df, district_col="district",
                   count_cols=("total_complaints", "nan_complaints"),
                   pop_col="population_2021", scale=1000):
    """Merge population and add per-scale-population columns."""
    pop = pop_df[["district", pop_col]].copy()
    pop["district"] = pop["district"].astype(str).str.strip()
    out = table.copy()
    out[district_col] = out[district_col].astype(str).str.strip()
    out = out.merge(pop.rename(columns={"district": district_col}),
                    on=district_col, how="left")
    for col in count_cols:
        if col in out.columns:
            out[f"{col}_per_{scale}"] = (
                out[col] / out[pop_col] * scale).round(2)
    return out
 
 
def percapita_with_aggregate(frame, nan_frame, pop_df):
    """District per-capita NaN — all years + Aggregate pivot."""
    long, _ = nan_dist_with_aggregate(frame, "district",
                                      nan_frame=nan_frame)
    long_pc  = add_per_capita(long, pop_df,
                               count_cols=("total_complaints",
                                           "nan_complaints"))
    agg_pc   = (long_pc[long_pc["custom_year"] == "Aggregate"]
                .sort_values("total_complaints_per_1000",
                             ascending=False).reset_index(drop=True))
 
    pc_cols    = ["total_complaints", "nan_complaints", "nan_pct",
                  "total_complaints_per_1000", "nan_complaints_per_1000"]
    year_order = VALID_YEARS + ["Aggregate"]
    pc_pivot   = long_pc.pivot_table(
        index="district", columns="custom_year",
        values=[c for c in pc_cols if c in long_pc.columns],
        aggfunc="sum")
    pc_pivot = pc_pivot.swaplevel(0, 1, axis=1)
    pc_pivot = pc_pivot.reindex(
        columns=pd.MultiIndex.from_product(
            [year_order,
             [c for c in pc_cols if c in long_pc.columns]]),
        fill_value=0)
    for col in pc_pivot.columns:
        pc_pivot[col] = pc_pivot[col].round(2)
    return agg_pc, pc_pivot
 
 
# ----- concentration ---------------------------------------------------------
 
def concentration_table(frame, group_cols, label="count"):
    """Unique ticket count + share_pct by group_cols, sorted descending."""
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    out = (frame.groupby(group_cols)["ticket_no"]
           .nunique().reset_index(name=label))
    total = out[label].sum()
    out["share_pct"] = (out[label] / total * 100).round(2) if total else 0.0
    return out.sort_values(label, ascending=False).reset_index(drop=True)
 
 
# ----- top-N ranking ---------------------------------------------------------
 
def top_n_within_group(frame, group_cols, count_col, n=5):
    """Keep top-n rows by count_col within each group.
    Adds rank and share_pct (within group) columns."""
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    group_total = (frame.groupby(group_cols)[count_col].sum()
                   .reset_index(name="group_total"))
    out = frame.merge(group_total, on=group_cols, how="left")
    out["share_pct"] = (out[count_col] / out["group_total"] * 100).round(2)
    out["rank"] = (out.groupby(group_cols)[count_col]
                   .rank(method="first", ascending=False))
    return (out[out["rank"] <= n]
            .sort_values(group_cols + ["rank"])
            .assign(rank=lambda x: x["rank"].astype(int))
            .drop(columns="group_total")
            .reset_index(drop=True))
 
 
# ----- top-10 subcategory label pivot ----------------------------------------
 
def top10_subcat_pivot(frame, months, district=None):
    """Top-10 subcategories per month as 'subcat (count, share%)'."""
    d = frame.copy()
    d["year_month"]  = d["created_on"].dt.to_period("M").astype(str)
    d["subcategory"] = d["subcategory"].fillna("NaN").astype(str).str.strip()
    if district:
        d = d[d["district"] == district]
    d = d[d["year_month"].isin(months)]
    monthly_totals = (d.groupby("year_month")["ticket_no"]
                      .nunique().reset_index(name="monthly_total"))
    monthly_subcat = (d.groupby(["year_month", "subcategory"])["ticket_no"]
                      .nunique().reset_index(name="count"))
    monthly_subcat = monthly_subcat.merge(
        monthly_totals, on="year_month", how="left")
    monthly_subcat["share_pct"] = (
        monthly_subcat["count"] / monthly_subcat["monthly_total"] * 100
    ).round(0).astype(int)
    monthly_subcat["rank"] = (
        monthly_subcat.groupby("year_month")["count"]
        .rank(method="first", ascending=False))
    top10 = monthly_subcat[monthly_subcat["rank"] <= 10].copy()
    top10["label"] = (top10["subcategory"] + " ("
                      + top10["count"].astype(int).astype(str) + ", "
                      + top10["share_pct"].astype(str) + "%)")
    pivot = (top10.pivot(index="year_month", columns="rank", values="label")
                  .sort_index())
    pivot.columns = [f"Rank {int(c)}" for c in pivot.columns]
    return pivot.T.fillna("nan")
 
 
# ----- yearly subcat pivot ---------------------------------------------------
 
def yearly_subcat_pivot(frame, mask):
    """Subcategory x year count + pct pivot for a masked frame."""
    d = frame[mask].copy()
    d["subcategory"] = (d["subcategory"].fillna("NaN")
                        .astype(str).str.strip())
    yearly = (d.groupby(["subcategory", "custom_year"])["ticket_no"]
               .nunique().reset_index(name="count"))
    year_totals = d.groupby("custom_year")["ticket_no"].nunique()
    yearly["total_cases"] = yearly["custom_year"].map(year_totals)
    yearly["percentage"]  = (
        yearly["count"] / yearly["total_cases"] * 100).round(2)
    return (yearly.pivot(index="subcategory", columns="custom_year",
                         values=["count", "percentage"])
                  .swaplevel(0, 1, axis=1).sort_index(axis=1).fillna(0))

## Prepare NaN subsets

In [6]:
df_nan_all = df[nan_mask(df)].copy()
df_focus   = df[df["custom_year"] == FOCUS_YEAR].copy()
df_nan_fy  = df_focus[nan_mask(df_focus)].copy()
pop_df     = population_df()
 
print(f"NaN subcategory (all years): {df_nan_all['ticket_no'].nunique():,}")
print(f"NaN subcategory (focus yr) : {df_nan_fy['ticket_no'].nunique():,}")

NaN subcategory (all years): 197,952
NaN subcategory (focus yr) : 122,991


## 1. Monthly NaN Distribution (All Years)

Monthly counts: total complaints, NaN-subcategory complaints, and the NaN
share, for every month in the analysis window.

In [7]:
monthly_nan_dist = nan_dist_table(
    df, "year_month", nan_frame=df_nan_all
).sort_values("year_month").reset_index(drop=True)
display(monthly_nan_dist)

,year_month,total_complaints,nan_complaints,nan_pct
0,2021-07,4651,29,0.62
1,2021-08,6637,27,0.41
2,2021-09,7172,35,0.49
3,2021-10,7262,77,1.06
4,2021-11,8614,1404,16.30
5,2021-12,8158,1104,13.53
6,2022-01,6912,1006,14.55
7,2022-02,4189,655,15.64
8,2022-03,5854,1049,17.92
9,2022-04,10222,1694,16.57


## 2. Yearly NaN Distribution

In [8]:
yearly_nan_dist = nan_dist_table(
    df, "custom_year", nan_frame=df_nan_all
).sort_values("custom_year").reset_index(drop=True)
display(yearly_nan_dist)

,custom_year,total_complaints,nan_complaints,nan_pct
0,2021-2022,98258,11167,11.36
1,2022-2023,181702,27065,14.90
2,2023-2024,355579,36729,10.33
3,2024-2025,688301,122991,17.87


## 3. District NaN Distribution

District-level totals, NaN counts, and NaN share 

In [9]:
district_nan_long, district_nan_pivot = nan_dist_with_aggregate(
    df, "district", nan_frame=df_nan_all)
 
print("District NaN — Focus Year (sorted by nan_pct):")
display(district_nan_long[district_nan_long["custom_year"] == FOCUS_YEAR]
        .sort_values("nan_pct", ascending=False).reset_index(drop=True))
 
print("\nDistrict NaN — All Years + Aggregate Pivot:")
display(district_nan_pivot)

District NaN — Focus Year (sorted by nan_pct):


,district,custom_year,total_complaints,nan_complaints,nan_pct
0,Nayagarh,2024-2025,31527,16593,52.63
1,Sambalpur,2024-2025,48029,21199,44.14
2,NaN,2024-2025,2113,709,33.55
3,Dhenkanal,2024-2025,29370,7413,25.24
4,Rayagada,2024-2025,6840,1551,22.68
5,Balangir,2024-2025,27665,5567,20.12
6,Jharsuguda,2024-2025,5739,1105,19.25
7,Jajpur,2024-2025,37942,6772,17.85
8,Baleswar,2024-2025,34184,5761,16.85
9,Kendrapara,2024-2025,24823,3977,16.02



District NaN — All Years + Aggregate Pivot:


2021-2022                               2022-2023  \
               total_complaints nan_complaints nan_pct total_complaints   
district                                                                  
Angul                      2759            422   15.30             3649   
Balangir                   4094            370    9.04            12428   
Baleswar                   7749           1054   13.60            15840   
Bargarh                    3426            443   12.93             4521   
Bhadrak                    4943            588   11.90             8312   
Boudh                       778            180   23.14             4286   
Cuttack                    4291            486   11.33             7592   
Deogarh                     594             66   11.11              944   
Dhenkanal                  4330            943   21.78             5855   
Gajapati                   1424            105    7.37             3996   
Ganjam                     5010            635   12.67            12624   
Jagatsinghapur             3268            271    8.29             7979   
Jajpur                     6042           1060   17.54             9885   
Jharsuguda                  742            104   14.02             1284   
Kalahandi                  9406            834    8.87            13137   
Kandhamal                  2444            167    6.83             2362   
Kendrapara                 4846            371    7.66             7673   
Kendujhar                  2082            164    7.88             3939   
Khordha                    8016            872   10.88            10198   
Koraput                    1372            169   12.32             2223   
Malkangiri                  472             69   14.62              494   
Mayurbhanj                 2812            194    6.90             6826   
NaN                        3872             44    1.14              599   
Nabarangpur                1028            112   10.89             3319   
Nayagarh                   2460            211    8.58             6446   
Nuapada                    1249             83    6.65             3243   
Puri                       3195            388   12.14             8542   
Rayagada                   1171            135   11.53             2491   
Sambalpur                  1573            235   14.94             2524   
Subarnapur                 1260            153   12.14             2662   
Sundargarh                 1550            239   15.42             5829   

                                             2023-2024                         \
               nan_complaints nan_pct total_complaints nan_complaints nan_pct   
district                                                                        
Angul                     615   16.85             8703            963   11.07   
Balangir                 1879   15.12            25275           2233    8.83   
Baleswar                 3185   20.11            22134           3876   17.51   
Bargarh                   684   15.13            10264            928    9.04   
Bhadrak                  1148   13.81            14256           1646   11.55   
Boudh                     577   13.46             4406            336    7.63   
Cuttack                  1142   15.04            13828           1894   13.70   
Deogarh                   110   11.65             3091            157    5.08   
Dhenkanal                1941   33.15            15960           3504   21.95   
Gajapati                  436   10.91             5938            346    5.83   
Ganjam                   1961   15.53            24319           2584   10.63   
Jagatsinghapur            501    6.28             8698            620    7.13   
Jajpur                   2287   23.14            12807           1763   13.77   
Jharsuguda                299   23.29             4347            569   13.09   
Kalahandi                1314   10.00            22034           1312    5.95   
Kandhamal    

## 4. District Per-Capita NaN

In [11]:
district_pc_agg, district_pc_pivot = percapita_with_aggregate(
    df, df_nan_all, pop_df)
 
print("\nDistrict Per-Capita NaN — All Years + Aggregate Pivot:")
display(district_pc_pivot)


District Per-Capita NaN — All Years + Aggregate Pivot:


2021-2022                         \
               total_complaints nan_complaints nan_pct   
district                                                 
Angul                      2759            422   15.30   
Balangir                   4094            370    9.04   
Baleswar                   7749           1054   13.60   
Bargarh                    3426            443   12.93   
Bhadrak                    4943            588   11.90   
Boudh                       778            180   23.14   
Cuttack                    4291            486   11.33   
Deogarh                     594             66   11.11   
Dhenkanal                  4330            943   21.78   
Gajapati                   1424            105    7.37   
Ganjam                     5010            635   12.67   
Jagatsinghapur             3268            271    8.29   
Jajpur                     6042           1060   17.54   
Jharsuguda                  742            104   14.02   
Kalahandi                  9406            834    8.87   
Kandhamal                  2444            167    6.83   
Kendrapara                 4846            371    7.66   
Kendujhar                  2082            164    7.88   
Khordha                    8016            872   10.88   
Koraput                    1372            169   12.32   
Malkangiri                  472             69   14.62   
Mayurbhanj                 2812            194    6.90   
NaN                        3872             44    1.14   
Nabarangpur                1028            112   10.89   
Nayagarh                   2460            211    8.58   
Nuapada                    1249             83    6.65   
Puri                       3195            388   12.14   
Rayagada                   1171            135   11.53   
Sambalpur                  1573            235   14.94   
Subarnapur                 1260            153   12.14   
Sundargarh                 1550            239   15.42   

                                                                  \
               total_complaints_per_1000 nan_complaints_per_1000   
district                                                           
Angul                               1.98                    0.30   
Balangir                            2.32                    0.21   
Baleswar                            3.01                    0.41   
Bargarh                             2.22                    0.29   
Bhadrak                             2.95                    0.35   
Boudh                               1.59                    0.37   
Cuttack                             1.53                    0.17   
Deogarh                             1.74                    0.19   
Dhenkanal                           3.43                    0.75   
Gajapati                            2.31                    0.17   
Ganjam                              1.31                    0.17   
Jagatsinghapur                      2.75                    0.23   
Jajpur                              3.03                    0.53   
Jharsuguda                          1.19                    0.17   
Kalahandi                           5.37                    0.48   
Kandhamal                           3.03                    0.21   
Kendrapara                          3.18                    0.24   
Kendujhar                           1.05                    0.08   
Khordha                             3.08                    0.34   
Koraput                             0.91                    0.11   
Malkangiri                          0.68                    0.10   
Mayurbhanj                          1.02                    0.07   
NaN                                 0.00                    0.00   
Nabarangpur                         0.74                    0.08   
Nayagarh                            2.46                    0.21   
Nuapada                             1.91                    0.13   
Puri                                1.74                    0.21   
Rayagada 

## 5. District x Month NaN Distribution (2024-25)

Long and wide views of NaN share by district and month within the focus
year.

In [14]:
district_month_nan_dist = nan_dist_table(
    df_focus, ["year_month", "district"], nan_frame=df_nan_fy
).sort_values(["year_month", "nan_pct"],
              ascending=[True, False]).reset_index(drop=True)
 
pivot = nan_dist_pivot(district_month_nan_dist, "district", "year_month")
display(pivot)

year_month            2024-07                                 2024-08          \
               nan_complaints nan_pct total_complaints nan_complaints nan_pct   
district                                                                        
Angul                     261   12.97             2013            209   12.68   
Balangir                 1299   19.52             6656            980   21.72   
Baleswar                  595   14.13             4212            244    7.29   
Bargarh                   314   13.94             2252            140    6.94   
Bhadrak                   440   10.95             4020            220    5.66   
Boudh                     153   19.29              793             51    9.11   
Cuttack                   639   17.52             3648            505   14.20   
Deogarh                    52    9.92              524             28    3.05   
Dhenkanal                 427   13.92             3067            176    7.04   
Gajapati                  147   27.84              528             69    8.78   
Ganjam                    868   18.04             4812            587   12.86   
Jagatsinghapur            127    8.77             1448             88    6.61   
Jajpur                    643   24.95             2577            405   15.91   
Jharsuguda                167   25.57              653             66   13.72   
Kalahandi                 966   17.75             5441            552   11.46   
Kandhamal                  99    7.01             1413             80    6.32   
Kendrapara                481   18.17             2647            557   25.19   
Kendujhar                 430   17.44             2465            140    8.04   
Khordha                   658   16.13             4080            411   11.54   
Koraput                   124    8.82             1406             81    4.86   
Malkangiri                 57   13.60              419             40   13.47   
Mayurbhanj                488   14.77             3304            228    9.74   
NaN                        47   28.66              164             22   17.19   
Nabarangpur               242   18.86             1283            111    8.14   
Nayagarh                  354   19.30             1834            354   27.06   
Nuapada                   162   12.16             1332             39    3.41   
Puri                      334   15.82             2111            222    9.40   
Rayagada                  227   27.28              832            131   23.48   
Sambalpur                1673   30.06             5566           2816   39.77   
Subarnapur                106    9.84             1077             38    5.13   
Sundargarh                215   11.94             1800            132   10.38   

year_month                             2024-09                           \
               total_complaints nan_complaints nan_pct total_complaints   
district                                                                  
Angul                      1648            262   10.51             2492   
Balangir                   4513            634   17.76             3570   
Baleswar                   3349            591   15.21             3886   
Bargarh                    2016            200    5.77             3466   
Bhadrak                    3885            443   10.26             4318   
Boudh                       560             80   13.58              589   
Cuttack                    3556            478   12.18             3926   
Deogarh                     917             70    6.86             1021   
Dhenkanal                  2500           1037   29.71             3490   
Gajapati                    786            138   14.65              942   
Ganjam                     4564            894   16.58             5392   
Jagatsinghapur             1331            136    9.04             1505   
Jajpur                     2545           1350   30.16             4476   
Jharsuguda                  481            17

## 6. District x Block Concentration (2024-25)

Where the NaN-subcategory complaints concentrate by district and block.

In [16]:
block_dist = concentration_table(
    df_nan_fy, ["district", "block"], label="count")
display(block_dist.head(30))

,district,block,count,share_pct
0,Sambalpur,Kuchinda,19815,16.11
1,Nayagarh,Odagaon,12773,10.39
2,Jajpur,Jajpur,1322,1.07
3,Dhenkanal,Parjang,1233,1.00
4,Jajpur,Bari,1187,0.97
5,Cuttack,Narasinghpur,1183,0.96
6,Dhenkanal,Hindol,1101,0.90
7,Dhenkanal,Kamakhyanagar,1068,0.87
8,Khordha,Bhubaneswar mc,1016,0.83
9,Dhenkanal,Odapada,993,0.81


## 7. Petitioner Concentration (2024-25)

Which petitioners (name x district x block) file the most NaN complaints.

In [17]:
if "petitioner_name" in df_nan_fy.columns:
    petitioner_dist = concentration_table(
        df_nan_fy, ["district", "block", "petitioner_name"], label="count")
    display(petitioner_dist.head(30))
else:
    petitioner_dist = pd.DataFrame()
    print("petitioner_name not present; skipping.")

,district,block,petitioner_name,count,share_pct
0,Sambalpur,Kuchinda,Samir Nayak,19599,15.94
1,Nayagarh,Odagaon,ପବିତ୍ର କୁମାର ଦାଶ,9835,8.00
2,Nayagarh,Odagaon,PABITRA KUMAR DASH,1540,1.25
3,Jagatsinghapur,Biridi,SWADHIN RANJAN DAS,751,0.61
4,Baleswar,Bahanaga,Rajeev Kumar Tiwari,327,0.27
5,Bhadrak,Chandabali,Suresh Chandra Nayak,314,0.26
6,Kendrapara,Aul,Sunakar Sahoo,280,0.23
7,NaN,NaN,mENtor pAWan,253,0.21
8,Kendrapara,Mahakalpara,Sarbeswar Bai,180,0.15
9,NaN,NaN,ପବିତ୍ର କୁମାର ଦାଶ,147,0.12


## 8. Sambalpur Top-10 Subcategories during the NaN Spike

The original analysis flagged a NaN spike in Sambalpur during June-October
2024. Below: each month's top-10 subcategories formatted as
"subcategory (count, share%)".

In [19]:
MONTHS_SPIKE  = ["2024-06", "2024-07", "2024-08", "2024-09", "2024-10"]
pivot_top10_T = top10_subcat_pivot(df, MONTHS_SPIKE, district="Sambalpur")
display(pivot_top10_T)

year_month,2024-06,2024-07,2024-08,2024-09,2024-10
Rank 1,"IAY/MKY/BPGY/PMAY (594, 50%)","NaN (1673, 30%)","NaN (2816, 40%)","NaN (4562, 49%)","NaN (6754, 62%)"
Rank 2,"Rural Housing (258, 22%)","IAY/MKY/BPGY/PMAY (1338, 24%)","Allegation/Corruption (2004, 28%)","Allegation/Corruption (1329, 14%)","Allegation/Corruption (1449, 13%)"
Rank 3,"NaN (70, 6%)","Allegation/Corruption (1010, 18%)","IAY/MKY/BPGY/PMAY (910, 13%)","Others (782, 8%)","IAY/MKY/BPGY/PMAY (1077, 10%)"
Rank 4,"Others (58, 5%)","Others (525, 9%)","Others (595, 8%)","Illegal Activities (612, 7%)","Rural Housing (611, 6%)"
Rank 5,"Allegation/Corruption (47, 4%)","Rural Housing (277, 5%)","Miscellaneous/Others (110, 2%)","IAY/MKY/BPGY/PMAY (609, 6%)","Others (303, 3%)"
Rank 6,"Miscellaneous/Others (26, 2%)","Miscellaneous/Others (139, 2%)","Rural Housing (99, 1%)","Rural Housing (502, 5%)","Miscellaneous/Others (283, 3%)"
Rank 7,"Service Matters (14, 1%)","Ration Card Issue (51, 1%)","Illegal Activities (90, 1%)","Miscellaneous/Others (336, 4%)","Police Cases (35, 0%)"
Rank 8,"OAP/ODP/MBPY/WP (11, 1%)","Road (48, 1%)","Ration Card Issue (51, 1%)","Higher Education (205, 2%)","Road (34, 0%)"
Rank 9,"Water Supply issue (8, 1%)","OAP/ODP/MBPY/WP (41, 1%)","Road (51, 1%)","Police Cases (47, 1%)","Ration Card Issue (25, 0%)"
Rank 10,"Land Allotment/Settlement/RoR (7, 1%)","Service Matters (29, 1%)","Land Allotment/Settlement/RoR (34, 0%)","Employment/Recruitment (36, 0%)","Land Allotment/Settlement/RoR (18, 0%)"


## 9. NaN Subcategory Rank and Yearly Pivot

In [20]:
cat_nan_mask = df["category"].isna() | (
    df["category"].astype(str).str.strip().isin(["", "NaN"]))
 
rank_subcat_nan = (
    concentration_table(df[cat_nan_mask], "subcategory", label="count")
    .rename(columns={"count": "count", "share_pct": "share_pct"}))
display(rank_subcat_nan.head(30))
 
pivot_nan = yearly_subcat_pivot(df, cat_nan_mask)
display(pivot_nan.head(30))

,subcategory,count,share_pct
0,NaN,197936,90.89
1,Excise Matters,19792,9.09
2,Water Supply issue,37,0.02
3,Test Sub Category,1,0.00


custom_year        2021-2022            2022-2023            2023-2024  \
                       count percentage     count percentage     count   
subcategory                                                              
Excise Matters       19792.0      63.89       0.0       0.00       0.0   
NaN                  11156.0      36.01   27060.0      99.98   36729.0   
Test Sub  Category       0.0       0.00       1.0       0.00       0.0   
Water Supply issue      32.0       0.10       4.0       0.01       1.0   

custom_year                   2024-2025             
                   percentage     count percentage  
subcategory                                         
Excise Matters            0.0       0.0        0.0  
NaN                     100.0  122991.0      100.0  
Test Sub  Category        0.0       0.0        0.0  
Water Supply issue        0.0       0.0        0.0